# 03 — Model Training

Chronological 75/25 train/test split. 5 models trained and evaluated.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import yaml
from pathlib import Path

cfg = yaml.safe_load(open('../configs/config.yaml'))
df = pd.read_csv('../data/processed/hourly_processed.csv', index_col=0, parse_dates=True)
df.index = pd.to_datetime(df.index, utc=True)
df = df.sort_index()
print(f"Loaded: {df.shape}")


In [ ]:
from src.features.build_features import build_features, get_feature_cols
from src.models.train import chronological_split

lag_hours = cfg['features']['lag_hours']
rolling_windows = cfg['features']['rolling_windows']
horizon = cfg['target']['horizon_hours']
train_frac = cfg['model']['train_fraction']

feat = build_features(df, lag_hours=lag_hours, rolling_windows=rolling_windows, horizon=horizon)
feature_cols = get_feature_cols(feat)

train, test = chronological_split(feat, train_frac)
print(f"Train: {len(train):,} rows ({train.index.min().date()} --> {train.index.max().date()})")
print(f"Test : {len(test):,} rows ({test.index.min().date()} --> {test.index.max().date()})")

X_train = train[feature_cols]
y_train = train['target']
X_test = test[feature_cols]
y_test = test['target']


## Model Training

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from src.evaluation.metrics import evaluate_all, print_metrics
from src.models.train import (baseline_persistence, baseline_seasonal_naive,
                               make_ml_pipeline)

seed = cfg['model']['random_seed']
results = []

# Baselines
for name, pred_fn in [('Persistence', baseline_persistence),
                       ('Seasonal Naive', baseline_seasonal_naive)]:
    y_pred = pred_fn(X_test)
    m = evaluate_all(y_test.values, y_pred.values, name)
    results.append(m)
    print_metrics(m)

# ML models
models = {
    'Linear Regression': make_ml_pipeline(LinearRegression(), use_scaler=True),
    'Random Forest': make_ml_pipeline(
        RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1, random_state=seed)),
    'HistGradientBoosting': make_ml_pipeline(
        HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=6, random_state=seed)),
}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = pd.Series(model.predict(X_test), index=X_test.index)
    m = evaluate_all(y_test.values, y_pred.values, name)
    results.append(m)
    print_metrics(m)


In [ ]:
results_df = pd.DataFrame(results)
print("\nModel Comparison Table:")
with pd.option_context('display.float_format', '{:.4f}'.format):
    print(results_df.to_string(index=False))
print()
best = results_df[~results_df['model'].isin(['Persistence','Seasonal Naive'])].loc[
    results_df[~results_df['model'].isin(['Persistence','Seasonal Naive'])]['MAE'].idxmin()]
print(f"Best ML model: {best['model']} (MAE={best['MAE']:.4f})")
